# Secrets Management

In [ ]:
# Day 49 — Secrets Management
# Rule #1 of DevOps & LLMOps security:
# NEVER hardcode API keys, DB passwords, or tokens in Python code or Git!
#
# In AI/MLOps, we juggle sensitive keys daily:
#   - OpenAI / Anthropic / Azure OpenAI keys
#   - HuggingFace tokens
#   - MLflow / DVC remote storage credentials
#
# Today we'll cover:
#   1. Local dev with `python-dotenv` (.env + .gitignore)
#   2. Azure Key Vault SDK (`azure-keyvault-secrets` + `DefaultAzureCredential`)
#   3. HashiCorp Vault (`hvac`)
#   4. Secrets best practices (never log secrets, env-aware loading)
#
# Install: pip install python-dotenv azure-keyvault-secrets azure-identity hvac

### 1. Local Development with `python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv

# Write a sample .env file (in real repos, ALWAYS add .env to .gitignore!)
with open(".env.example", "w") as f:
    f.write("OPENAI_API_KEY=sk-example-replace-me\nENVIRONMENT=dev\n")

# Load variables from .env into os.environ
load_dotenv(".env.example")

api_key = os.getenv("OPENAI_API_KEY")
env = os.getenv("ENVIRONMENT", "dev")

# NEVER print the full secret! Mask it in logs.
masked = api_key[:5] + "***" if api_key else "MISSING"
print(f"Environment: {env} | Key loaded: {masked}")

### 2. Production Secrets: Azure Key Vault & HashiCorp Vault

In [ ]:
# In production, we don't ship .env files into Docker containers.
# We fetch secrets dynamically using Managed Identity / IAM / Vault tokens.

azure_kv_code = '''from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

vault_url = "https://my-mlops-vault.vault.azure.net/"
credential = DefaultAzureCredential()
client = SecretClient(vault_url=vault_url, credential=credential)

# Read secret
openai_key = client.get_secret("openai-api-key").value

# Rotate / update secret
client.set_secret("openai-api-key", "sk-new-rotated-key")
'''

hvac_code = '''import hvac

client = hvac.Client(url="https://vault.internal:8200", token=os.getenv("VAULT_TOKEN"))
secret = client.secrets.kv.v2.read_secret_version(path="mlops/openai")
api_key = secret["data"]["data"]["api_key"]
'''

print("--- Azure Key Vault Pattern ---")
print(azure_kv_code)
print("--- HashiCorp Vault Pattern ---")
print(hvac_code)

In [ ]:
# Practice Questions:
#
# 1. Why should `.env` NEVER be committed to Git, while `.env.example` SHOULD be?
# 2. How does `DefaultAzureCredential` work locally vs inside an AKS pod?
# 3. Why should you mask secrets in structured JSON logs?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Environment-Aware Secret Resolver
# Write a class `SecretManager` that:
# - Uses `.env` / `os.getenv` when `ENVIRONMENT == 'dev'`
# - Uses `SecretClient` (Azure Key Vault) when `ENVIRONMENT == 'prod'`

# Exercise 2: Secret Leak Scanner
# Write a pre-commit Python script that scans `.py` and `.yaml` files
# for regex patterns like `sk-[A-Za-z0-9]{20,}` and exits with code 1 if found.

# Exercise 3: Key Rotation Helper
# Write a function `rotate_secret(client, name, new_value)` that checks
# if the new value differs from the current secret before creating a new version.